# Exercise 8 — Sound transformations (HPS)

In this exercise you apply **time and frequency transformations** to sounds analyzed with the HPS model. The HPS representation — harmonic tracks (frequency, magnitude, phase) plus a smooth stochastic envelope — can be modified analytically before resynthesis, allowing musically meaningful transformations with no time-domain artefacts.

This exercise has **two parts**:
1. Apply a natural-sounding transformation to the `speech-female.wav` sound (same sound as E7).
2. Apply a creative transformation to a harmonic sound of your own choice.

**Transformation parameters:**
- **`freqScaling`**: time–value pairs; the value is a multiplicative factor applied uniformly to all harmonic frequencies. A value of `2` transposes up one octave; `0.5` transposes down one octave. Example: `[0, 2, 1, 2]` transposes the whole sound up one octave.
- **`freqStretching`**: time–value pairs; the value is a multiplicative factor whose effect grows with harmonic number — it makes the sound progressively inharmonic by stretching the upper partials away from perfect harmonicity. A value of `1` leaves the spectrum unchanged; `1.2` produces a strong metallic/bell-like inharmonic effect. Example: `[0, 1.2, 1, 1.2]`.
- **`timbrePreservation`**: `1` to preserve the original spectral envelope (formants) even when fundamental frequency changes, `0` to let the envelope shift with the pitch. Use `1` for speech transformations to retain speaker identity.
- **`timeScaling`**: time–value pairs; the value is the output time as a fraction of the input time. A value of `2` at the end stretches to double duration; `0.5` compresses to half. Example: `[0, 0, 1, 2]` stretches the full sound to twice its original duration.

All parameter arrays must be flat arrays of even length (interleaved time–value pairs). Times can be normalised (0–1) or in seconds, but must match the input sound duration. Multiple time points allow time-varying transformations, e.g. `[0, 1.2, 2.01, 1.2, 2.679, 0.7, 3.146, 0.7]`.

**Analysis tip for transformations:** An analysis optimised for faithful reconstruction (E7) is not always optimal for transformation. For strong transformations, prioritise **smooth, continuous harmonic tracks** and a **compact stochastic envelope** (low `stocf`). This may mean slightly relaxing `t` or `f0et` to avoid gaps in the harmonic tracks, even if the un-transformed resynthesis sounds slightly different from the original.

## Part 1 – Natural-sounding transformation of speech

Use `speech-female.wav` (from the `sounds/` directory) with the HPS model to produce a **natural-sounding transformation**: the output must still sound like speech a human could have produced, and should be perceptually as different as possible from the original while remaining intelligible or at least plausible as speech.

**Step 1 (E8-1.1):** Perform an HPS analysis/synthesis of the sound. You can reuse the parameters from E7, but note that analyses optimised for transformation quality may differ from those optimised for reconstruction fidelity — smooth, uninterrupted harmonic tracks and compact stochastic data are more important here than exact spectral accuracy.

**Step 2 (E8-1.3):** Apply time and/or frequency transformations to the analysis representation. Only a single transformation pass is allowed (no mixing of outputs from different transformations). Set `freqScaling`, `freqStretching`, `timbrePreservation`, and `timeScaling` to achieve the desired effect.

Write a short paragraph for each step: describe what perceptual change you aimed for, list all analysis and transformation parameter values used, and explain the reasoning behind each choice.

In [ ]:
import numpy as np
from scipy.signal import get_window
import matplotlib.pyplot as plt
import IPython.display as ipd

from smstools.models import utilFunctions as UF
from smstools.models import stft as STFT
from smstools.models import hpsModel as HPS
from smstools.transformations import hpsTransformations as HPST
from smstools.transformations import harmonicTransformations as HT

In [ ]:
# E8 - 1.1: perform HPS analysis and synthesis of speech-female.wav

input_file = '../sounds/speech-female.wav'

### set the parameters
window = 'XXX'
M = XXX
N = XXX
t = XXX
minSineDur = XXX
nH = XXX
minf0 = XXX
maxf0 = XXX
f0et = XXX
harmDevSlope = XXX
stocf = XXX

# no need to modify anything after this
Ns = 512
H = 128

(fs, x) = UF.wavread(input_file)
w = get_window(window, M, fftbins=True)
hfreq, hmag, hphase, stocEnv = HPS.hpsModelAnal(x, fs, w, N, H, t, nH, minf0, maxf0, f0et, harmDevSlope, minSineDur, Ns, stocf)
y, yh, yst = HPS.hpsModelSynth(hfreq, hmag, hphase, stocEnv, Ns, H, fs)

ipd.display(ipd.Audio(data=x, rate=fs))
ipd.display(ipd.Audio(data=y, rate=fs))

**E8 – 1.2: Analysis parameter justification (answer here)**

1. Compare the analysis parameters you used here with the ones you used in E7 for the same speech sound. Did you change any of them? If so, which parameters did you change, in which direction, and why does the transformation goal (smooth harmonic tracks, compact stochastic) require different values than the reconstruction goal (accurate spectrum)?
2. Listen to the resynthesis from E8-1.1 before applying any transformation. Is the reconstruction slightly worse than in E7? If so, is that an acceptable trade-off for transformation quality, and how would you detect whether a harmonic track has an unacceptable gap or discontinuity?
3. What value of `stocf` did you choose for this analysis, and why is a compact (smooth, low-dimensional) stochastic envelope particularly important when you intend to apply time-scaling to the residual?

In [ ]:
# E8 - 1.3: apply HPS transformation to speech-female.wav analysis

### define the transformations (time-value pairs as flat arrays)
freqScaling = np.array([XX, ])
freqStretching = np.array([XX, ])
timbrePreservation = X
timeScaling = np.array([XX, ])

# no need to modify anything after this
Ns = 512
H = 128

# frequency scaling of the harmonics
hfreqt, hmagt = HT.harmonicFreqScaling(hfreq, hmag, freqScaling, freqStretching, timbrePreservation, fs)

# time scaling the sound
yhfreq, yhmag, ystocEnv = HPST.hpsTimeScale(hfreqt, hmagt, stocEnv, timeScaling)

# synthesis from the transformed hps representation
y, yh, yst = HPS.hpsModelSynth(yhfreq, yhmag, np.array([]), ystocEnv, Ns, H, fs)

ipd.display(ipd.Audio(data=y, rate=fs))

**E8 – 1.4: Transformation description (answer here)**

1. Describe the perceptual goal of your transformation in one sentence (e.g., "make the speech sound like a child", "slow down speaking rate while preserving pitch"). Then list the exact values you used for all four transformation arrays (`freqScaling`, `freqStretching`, `timbrePreservation`, `timeScaling`).
2. Why did you set `timbrePreservation` to the value you chose? For a pitch-shift transformation, what audible difference does setting `timbrePreservation = 1` vs. `timbrePreservation = 0` produce for speech — and which setting makes the result sound more like natural speech?
3. Listen to the transformed output. Does it still sound like speech? Describe the most noticeable perceptual effect. If there are artefacts (clicks, buzzes, discontinuities), identify which analysis parameter or transformation parameter is most likely responsible and suggest how you would fix it.
4. Is the transformation time-invariant (the same at every frame) or time-varying (different values at different times)? If time-invariant, propose a time-varying version of one of your transformation parameters that would produce a more interesting or musically expressive result.

## Part 2 – Creative transformation of a sound of your choice

Choose any short (≤ 5 s) **monophonic harmonic sound** from [Freesound](https://freesound.org) — any acoustic instrument, speech, or other harmonic natural sound. Convert it to mono, 44100 Hz, 16-bit WAV. Apply the most creative and interesting single-pass HPS transformation you can produce. The result does **not** need to sound natural; it just needs to be created with a single transformation pass (no mixing of outputs from different transformations).

**Step 1 (E8-2.1):** Perform an HPS analysis/synthesis of your chosen sound. Provide the Freesound URL and justify your analysis parameters.

**Step 2 (E8-2.3):** Apply your creative transformation. Set all four transformation parameter arrays and describe what effect you were aiming for and why you chose those values.

Write a short paragraph for each step, giving all parameter values in sufficient detail for an evaluator to exactly reproduce the analysis and transformation.

In [ ]:
# E8 - 2.1: perform HPS analysis and synthesis of chosen Freesound sound

input_file = 'XXX'

### set the parameters
window = 'XXX'
M = XXX
N = XXX
t = XXX
minSineDur = XXX
nH = XXX
minf0 = XXX
maxf0 = XXX
f0et = XXX
harmDevSlope = XXX
stocf = XXX

# no need to modify anything after this
Ns = 512
H = 128

(fs, x) = UF.wavread(input_file)
w = get_window(window, M, fftbins=True)
hfreq, hmag, hphase, stocEnv = HPS.hpsModelAnal(x, fs, w, N, H, t, nH, minf0, maxf0, f0et, harmDevSlope, minSineDur, Ns, stocf)
y, yh, yst = HPS.hpsModelSynth(hfreq, hmag, hphase, stocEnv, Ns, H, fs)

ipd.display(ipd.Audio(data=x, rate=fs))
ipd.display(ipd.Audio(data=y, rate=fs))

**E8 – 2.2: Analysis parameter justification (answer here)**

1. Provide the Freesound URL of your chosen sound and one sentence explaining why it is a good candidate for a creative transformation (e.g., its pitch range, brightness, harmonic density, or timbral character).
2. List all nine HPS analysis parameters you used for this sound (window type, `M`, `N`, `t`, `minSineDur`, `nH`, `minf0`, `maxf0`, `f0et`, `harmDevSlope`, `stocf`) and give a one-sentence justification for each value, referencing either a visible spectrogram feature or a listening test result.
3. Did you need to adjust any parameter compared to what you would have chosen for a faithful reconstruction, in order to get cleaner harmonic tracks for transformation? If so, what was the trade-off?

In [ ]:
# E8 - 2.3: apply creative HPS transformation to chosen sound analysis

### define the transformations (time-value pairs as flat arrays)
freqScaling = np.array([XX, ])
freqStretching = np.array([XX, ])
timbrePreservation = X
timeScaling = np.array([XX, ])

# no need to modify anything after this
Ns = 512
H = 128

# frequency scaling of the harmonics
hfreqt, hmagt = HT.harmonicFreqScaling(hfreq, hmag, freqScaling, freqStretching, timbrePreservation, fs)

# time scaling the sound
yhfreq, yhmag, ystocEnv = HPST.hpsTimeScale(hfreqt, hmagt, stocEnv, timeScaling)

# synthesis from the transformed hps representation
y, yh, yst = HPS.hpsModelSynth(yhfreq, yhmag, np.array([]), ystocEnv, Ns, H, fs)

ipd.display(ipd.Audio(data=y, rate=fs))

**E8 – 2.4: Creative transformation description (answer here)**

1. Describe in one sentence the creative concept behind your transformation — what sonic world were you trying to create (e.g., robotic, underwater, alien, bells, choir)? List the exact values of all four transformation arrays.
2. Explain the role of `freqStretching` in your transformation. If you used a value different from 1, at which harmonic number does the deviation from perfect harmonicity become perceptually noticeable, and does the result sound more metallic, bell-like, or something else?
3. Did you use a time-varying transformation (different values at different time points)? If so, describe the trajectory and explain how it shapes the evolution of the sound over time. If not, propose a time-varying version — write out the array values — that would make the transformation more dynamic or expressive.
4. Compare the creative freedom offered by the HPS transformation framework with what would be achievable by applying time-domain effects (reverb, pitch shift, time-stretch plug-ins) to the same original sound. What can the HPS approach do that standard time-domain effects cannot, and what are its limitations?